In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report


from sklearn.linear_model import LogisticRegression, LinearRegression


In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df=pd.read_csv('/content/karachi_properties_post_feature_selection.csv')

In [4]:
df['price']=df['price']*10_000_000

In [5]:
X=df.drop(columns=['price'])
y=df['price']

In [6]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [7]:
columnsto_encode=['age_category','parking_score','location_score','sub_location_score','property_type_encoded']

In [8]:
y_transformed=np.log1p(y)

In [9]:
preprocessor=ColumnTransformer(
   transformers=[
       ('num',StandardScaler(),["area","bedrooms","bathrooms","kitchen","total_floors","area_per_bedroom"]),
       ('cat',OneHotEncoder(drop='first'),columnsto_encode)
   ] ,
   remainder='passthrough'
)

In [10]:
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',SVR(kernel='rbf'))
])

In [11]:
kfold=KFold(n_splits=10,shuffle=True,random_state=42)
scores=cross_val_score(pipeline,X,y_transformed,scoring='r2')

In [12]:
scores.mean()

np.float64(0.9554585030889337)

In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [14]:
pipeline.fit(X_train,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['area', 'bedrooms',
                                                   'bathrooms', 'kitchen',
                                                   'total_floors',
                                                   'area_per_bedroom']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['age_category',
                                                   'parking_score',
                                                   'location_score',
                                                   'sub_location_score',
                                                   'property_type_encoded'])])),
                ('regressor', SVR())])

In [15]:
y_pred=pipeline.predict(X_test)

In [16]:
y_pred=np.expm1(y_pred)

In [17]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

11247822.143149631

In [18]:
y.describe()

,price
count,2.033100e+04
mean,7.069856e+07
std,1.009790e+08
min,1.500000e+06
25%,1.820000e+07
50%,3.800000e+07
75%,8.250000e+07
max,2.160000e+09
